In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.getOrCreate()

# 1) Define a schema explicitly (industry best practice to avoid type ambiguity)
schema = T.StructType([
    T.StructField("id",      T.IntegerType(),  nullable=False),
    T.StructField("name",    T.StringType(),   nullable=False),
    T.StructField("age",     T.IntegerType(),  nullable=True),
    T.StructField("salary",  T.DoubleType(),   nullable=True),
    T.StructField("country", T.StringType(),   nullable=True),
    T.StructField("dept",    T.StringType(),   nullable=True),
])

# 2) Sample data rows
data = [
    (1, "Asha",   26,  55000.0, "IN", "Engineering"),
    (2, "Rohit",  29,  72000.0, "IN", "Data"),
    (3, "Meera",  0,  48000.0, "US", "Support"),
    (4, "Karan",  31,  88000.0, "UK", "Data"),
    (5, "Vijay",  None, 60000.0, "IN", "HR"),
    (6, "Vikas",  45, 45600.0, "IN", "CJ"),
    (7, "Yashu",  0,  90000.0, "IN", "Data"),
    (8, "Harshal",  45, 908000.0, "China", "Data"),
    (9, "Rajesh",  35,  90000.0, "IN", "Data"),
    (10, "Rakesh",  78,  90000.0, "IN", "Data"),
    (11, "Vijay",  None, 60000.0, "IN", "HR"),
    (12, "Vikas",  45, 45600.0, "IN", "CJ"), 
    (13, "Yashu",  35,  90000.0, "IN", "Data"), 
    (14, "Harshal",  45, 908000.0, "China", "Data"), 
    (15, "Rajesh",  35,  90000.0, "IN", "Data"),      # null age example
]

# 3) Create the DataFrame
df = spark.createDataFrame(data, schema=schema)

# 4) Inspect
df.printSchema()
df.show(truncate=False)

In [0]:
# it will fail if the table already exists
df.createOrReplaceTempView("table")

In [0]:

from pyspark.sql.functions import col, when

df = df.withColumn(
    "adult",
    when(col("age").isNull(), "Novalue")
    .when(col("age") < 18, "No")
    .when(col("age") > 18, "Yes")
    .otherwise("Novalue")   # this will catch exactly age == 18
)

df.show()



In [0]:
from pyspark.sql.functions import col, when, lit


df = (
    df
    .withColumn(
        "age",
        when(col("age").isNull() | (col("age") == 0), lit(19))
        .otherwise(col("age"))
    )
    .withColumn(
        "adult",
        when(col("age") > 18, "Yes").otherwise("No")
    )
)

df.show()


In [0]:
spark.sql("""
          select * ,
          case when age<18 then  'minor'
          when age>18 then  'major'
          else 'novalue'
          end as  adult 

            from table
          """).show()